
# 06 — Player Style Similarity

## Decoding Winger Playstyles

Previous notebooks established a historical population of 1,396 winger
player-club-season observations and developed a 20-feature representation
of attacking behaviour.

Notebook 05 used PCA to understand the dominant dimensions of the
resulting playstyle space.

Although the first five principal components captured 85.23% of
standardized feature variance, the reduced representation substantially
changed nearest-neighbour relationships.

Consequently, the complete 20-feature standardized representation was
retained as the primary baseline for player-style similarity.

## Research Question

**Which winger player spells exhibit similar attacking behaviours,
and which specific behaviours explain their similarities and differences?**

The objective is to compare how footballers attack, independently of
their goals, assists, finishing efficiency or overall player quality.

Potential applications include:

- Identifying stylistically similar players.
- Finding possible replacements for specific attacking roles.
- Distinguishing players with similar output but different behaviour.
- Examining how an individual player's style changes between seasons.
- Explaining the features behind individual player comparisons.

## Analytical Principles

**1. Style is distinct from quality.**

Only the 20 retained behavioural features determine stylistic similarity.
Performance and output statistics will be considered separately.

**2. Similarity is multidimensional.**

Players are compared across the complete attacking-style representation,
not a single statistic or PCA component.

**3. Similarity must be explainable.**

A numerical similarity score is insufficient without examining which
behavioural features drive the comparison.

**4. Football interpretation remains essential.**

Close statistical neighbours must be evaluated using their observed
behavioural profiles. Statistical proximity alone does not establish
tactical interchangeability.

**5. Methodological choices must be tested.**

Scaling, weighting and distance metrics influence similarity rankings.
The primary method is treated as a baseline rather than assumed to be
objectively optimal.

## Observational Grain

Each observation represents a player × club × season spell.

Multiple seasons and different club spells are preserved because an
individual player's observed attacking role may change over time.

For cross-player similarity searches, other observations belonging
to the same player_id will be excluded from the candidate neighbours.


# 1. Data Loading and Validation

The similarity analysis uses the finalized outputs of Notebook 04.

### Primary Inputs

**Standardized behavioural matrix**

20 globally standardized attacking-style features.

This is the baseline representation for similarity analysis.

**Raw behavioural matrix**

The same features in their original football-interpretable units.

This will be used to explain individual player comparisons.

**Family-balanced matrix**

The alternative representation created in Notebook 04 using
behavioural-family weighting.

It will be used for sensitivity analysis rather than automatically
replacing the primary representation.

**Feature specification**

Contains the retained feature names, behavioural families and weights.

### Additional Data

Quality/output descriptors remain outside the similarity calculation.

They may be joined later to evaluate players who exhibit comparable
styles but differ in effectiveness or production.

The dataset and feature definitions are not modified in this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import pairwise_distances

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

OUTPUT_DIR = Path("/content/")

required_files = [
    "winger_style_features_raw.parquet",
    "winger_style_features_scaled.parquet",
    "winger_style_features_family_balanced.parquet",
    "winger_style_feature_specification.csv"
]

missing = [
    filename
    for filename in required_files
    if not (OUTPUT_DIR / filename).exists()
]

if missing:
    raise FileNotFoundError(
        f"Missing required files: {missing}"
    )

print("All Notebook 04 inputs available.")

In [ ]:
raw_df = pd.read_parquet(
    OUTPUT_DIR / "winger_style_features_raw.parquet"
)

scaled_df = pd.read_parquet(
    OUTPUT_DIR / "winger_style_features_scaled.parquet"
)

balanced_df = pd.read_parquet(
    OUTPUT_DIR / "winger_style_features_family_balanced.parquet"
)

feature_spec = pd.read_csv(
    OUTPUT_DIR / "winger_style_feature_specification.csv"
)

identity_cols = [
    "player",
    "player_id",
    "squad",
    "comp",
    "season",
    "age",
    "born",
    "Minutes"
]

features = feature_spec["feature"].tolist()

assert len(features) == 20
assert len(set(features)) == 20

expected_columns = identity_cols + features

for name, df_check in {
    "Raw": raw_df,
    "Scaled": scaled_df,
    "Balanced": balanced_df
}.items():

    assert df_check.shape == (1396, 28)
    assert df_check.columns.tolist() == expected_columns

    assert not df_check.duplicated(
        ["player_id", "squad", "season"]
    ).any()

    assert np.isfinite(
        df_check[features].to_numpy(dtype=float)
    ).all()

for df_check in [raw_df, balanced_df]:

    pd.testing.assert_frame_equal(
        scaled_df[identity_cols].reset_index(drop=True),
        df_check[identity_cols].reset_index(drop=True)
    )

# Primary feature representation
X = scaled_df[features].copy()

assert np.allclose(
    X.mean().to_numpy(), 0, atol=1e-8
)

assert np.allclose(
    X.std(ddof=0).to_numpy(), 1, atol=1e-6
)

print("Validation passed.")
print("Observations:", len(X))
print("Features:", X.shape[1])
print("Unique players:", scaled_df["player_id"].nunique())

display(
    feature_spec[
        ["feature", "macro_family", "family_weight"]
    ]
)


# 2. Constructing the Baseline Similarity Space

## 2.1 Distance Metric — Euclidean Distance

The primary similarity analysis begins with Euclidean distance
across the 20 standardized attacking-style features.

For two player spells A and B:

$$
d(A,B) =
\sqrt{\sum_{j=1}^{20}(z_{Aj}-z_{Bj})^2}
$$

where:

- \(z_{Aj}\) is player A's standardized value for feature \(j\).
- \(z_{Bj}\) is player B's standardized value for the same feature.
- \(d(A,B)\) is the overall distance between their attacking profiles.

### Interpretation

A smaller distance indicates more similar observed attacking
behaviour across the selected features.

A larger distance indicates greater behavioural dissimilarity.

Because the features are standardized, differences are measured
relative to the historical variation of each variable.

For example, a difference of one standard deviation in crossing
frequency contributes the same squared distance as a difference
of one standard deviation in shooting frequency.

### Why Euclidean Distance?

Euclidean distance provides:

1. A transparent mathematical definition of similarity.
2. Equal initial treatment of standardized features.
3. An exact decomposition into feature-level squared differences.
4. A straightforward baseline for evaluating alternative approaches.

However, Euclidean distance has limitations.

- Correlated features can reinforce particular behaviours.
- Large differences in a few features can strongly influence distance.
- Similarity depends on the chosen feature representation.
- Statistical proximity does not necessarily imply tactical
  interchangeability.

Euclidean distance is therefore selected as the initial baseline,
not assumed to be the final optimal metric.

### Same-Player Exclusion

When identifying stylistically similar footballers, observations
belonging to the same player_id are excluded from the candidate pool.

This prevents different seasons of the same player from dominating
the neighbour lists.

These observations remain available for later analysis of
individual player-style evolution.

In [ ]:
# Calculate Euclidean distances across all 20 features
distance_matrix = pairwise_distances(
    X.to_numpy(),
    metric="euclidean"
)

# Mask all spells belonging to the same footballer
player_ids = scaled_df["player_id"].to_numpy()

same_player = (
    player_ids[:, None] == player_ids[None, :]
)

distance_matrix[same_player] = np.inf

# Validate
assert distance_matrix.shape == (1396, 1396)

assert np.isinf(
    distance_matrix[same_player]
).all()

assert np.allclose(
    distance_matrix,
    distance_matrix.T
)

assert np.isfinite(
    distance_matrix[~same_player]
).all()

print("Euclidean distance matrix constructed.")
print("Shape:", distance_matrix.shape)
print("Same-player observations excluded.")


## 2.2 Distribution of Stylistic Distances

Euclidean distance provides a relative measure of stylistic
dissimilarity, not an inherently meaningful percentage score.

For example, a distance of 2 does not mean two players are
"twice as similar" as players separated by a distance of 4.

We therefore inspect the distribution of distances across
different-player observations.

This provides context for interpreting individual comparisons.

We examine:

- Distance distribution across all eligible player-spell pairs.
- Distance to the nearest eligible neighbour for each spell.
- How the nearest-neighbour distances compare with ordinary
  distances across the historical population.

These diagnostics establish a descriptive reference for
interpreting the similarity results.

They do not establish universal thresholds for what constitutes
a similar or dissimilar footballer.

In [ ]:
# Upper triangle avoids counting each pair twice
upper_triangle = np.triu_indices(
    len(scaled_df),
    k=1
)

all_distances = distance_matrix[upper_triangle]

# Exclude masked same-player pairs
all_distances = all_distances[
    np.isfinite(all_distances)
]

nearest_distances = np.min(
    distance_matrix,
    axis=1
)

distance_summary = pd.DataFrame({
    "statistic": [
        "Minimum",
        "5th percentile",
        "25th percentile",
        "Median",
        "75th percentile",
        "95th percentile",
        "Maximum"
    ],

    "all_pair_distances": [
        np.min(all_distances),
        np.percentile(all_distances, 5),
        np.percentile(all_distances, 25),
        np.median(all_distances),
        np.percentile(all_distances, 75),
        np.percentile(all_distances, 95),
        np.max(all_distances)
    ],

    "nearest_neighbour_distances": [
        np.min(nearest_distances),
        np.percentile(nearest_distances, 5),
        np.percentile(nearest_distances, 25),
        np.median(nearest_distances),
        np.percentile(nearest_distances, 75),
        np.percentile(nearest_distances, 95),
        np.max(nearest_distances)
    ]
})

display(distance_summary.round(3))

# Visualize the distribution of pairwise distances
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    all_distances,
    bins=50,
    alpha=0.8
)

ax.axvline(
    np.median(all_distances),
    linestyle="--",
    label="Median pairwise distance"
)

ax.axvline(
    np.median(nearest_distances),
    linestyle=":",
    label="Median nearest-neighbour distance"
)

ax.set_title(
    "Distribution of Winger Style Distances"
)

ax.set_xlabel(
    "Euclidean Distance (20 Standardized Features)"
)

ax.set_ylabel(
    "Number of Player-Spell Pairs"
)

ax.legend()
plt.tight_layout()
plt.show()


## 2.3 Findings — Population-Level Stylistic Distances

The distribution of Euclidean distances provides a reference for
interpreting subsequent player comparisons.

Across eligible different-player spell pairs:

- Median pairwise distance: 5.748
- Interquartile range: 4.596–7.155
- 95th-percentile pairwise distance: 9.646

The distance distribution is right-skewed, with a relatively
long tail of highly dissimilar player profiles.

Nearest-neighbour distances are substantially smaller:

- Median nearest-neighbour distance: 1.982
- 95th percentile: 2.889
- Maximum: 7.147

Most winger spells therefore have at least one substantially
closer alternative within the historical population.

However, some observations are considerably more isolated.

These values are descriptive reference points, not universal
thresholds of football similarity.

The next stage examines individual player comparisons to
understand how these distances translate into football behaviour.


# 3. Initial Player-Similarity Inspection

## 3.1 Identifying Nearest Stylistic Neighbours

We begin by examining three illustrative player-season spells:

1. Jérémy Doku — Manchester City, 2023–24.
2. Mohamed Salah — Liverpool, 2017–18.
3. Lionel Messi — Barcelona, 2018–19.

These observations represent contrasting attacking behaviours
identified in the PCA analysis.

For each target, the ten closest player spells are retrieved
from the historical population using standardized Euclidean distance.

Other observations belonging to the target footballer's
player_id are excluded.

### Questions

- Which players are identified as the closest stylistic neighbours?
- Are the results plausible from a football perspective?
- Do comparisons reflect similar attacking methods rather than
  merely similar attacking output?
- Are any results surprising or counterintuitive?
- Do certain distances require closer feature-level inspection?

At this stage, similarity rankings are treated as candidate
comparisons rather than validated conclusions.

In [ ]:
def find_similar_players(
    player,
    squad,
    season,
    k=10
):

    # Identify the exact player-club-season observation
    target_mask = (
        (scaled_df["player"] == player)
        &
        (scaled_df["squad"] == squad)
        &
        (scaled_df["season"] == season)
    )

    target_positions = np.flatnonzero(
        target_mask.to_numpy()
    )

    if len(target_positions) != 1:
        raise ValueError(
            f"Expected one match for {player}, "
            f"{squad}, {season}; "
            f"found {len(target_positions)}."
        )

    target_idx = target_positions[0]

    # Retrieve closest eligible observations
    nearest_idx = np.argsort(
        distance_matrix[target_idx]
    )[:k]

    neighbours = scaled_df.iloc[
        nearest_idx
    ][
        [
            "player",
            "squad",
            "season",
            "Minutes"
        ]
    ].copy()

    neighbours.insert(
        0,
        "rank",
        np.arange(1, k + 1)
    )

    neighbours["distance"] = (
        distance_matrix[target_idx, nearest_idx]
    )

    return neighbours.reset_index(drop=True)


test_players = [
    ("Jeremy Doku", "Manchester City", "2023-2024"),
    ("Mohamed Salah", "Liverpool", "2017-2018"),
    ("Lionel Messi", "Barcelona", "2018-2019")
]

for player, squad, season in test_players:

    print("\n" + "=" * 65)
    print(f"{player} — {squad} ({season})")
    print("=" * 65)

    neighbours = find_similar_players(
        player,
        squad,
        season
    )

    display(neighbours.round(3))


## 3.2 Initial Findings — Football Plausibility and Limitations

The initial nearest-neighbour results demonstrate both plausible
football relationships and important methodological limitations.

### Jérémy Doku — Manchester City, 2023–24

Doku's closest available match is Vinícius Júnior (2022–23),
at a Euclidean distance of 7.147.

This equals the maximum nearest-neighbour distance observed
across the population, to the reported precision.

His closest matches generally include direct, carry-oriented
attackers, supporting the broad football plausibility of the
rankings.

However, his extreme isolation means that his nearest
available player should not automatically be described as
a close stylistic equivalent.

### Mohamed Salah — Liverpool, 2017–18

The nearest observations include Ansu Fati, Michail Antonio,
Serge Gnabry, Iñaki Williams and Kylian Mbappé.

These candidates suggest that the model identifies some common
behavioural characteristics among direct attacking forwards.

Individual comparisons, particularly the less intuitive
matches, require feature-level inspection before stronger
football conclusions can be drawn.

### Lionel Messi — Barcelona, 2018–19

Messi's closest observations include several Neymar seasons,
alongside Alexis Sánchez, Philippe Coutinho and other
creative attacking players.

The broad candidate profiles appear plausible.

However, Messi also has an unusually large nearest-neighbour
distance compared with the population median.

### Methodological Observations

All three selected test cases have nearest-neighbour distances
above the population's 95th percentile.

They are therefore informative examples of unusual player
profiles, but not representative of typical similarity results.

Additionally, multiple seasons of the same candidate player
appear repeatedly in the top-ten rankings.

For recruitment-oriented comparisons, this reduces the
diversity of recommended footballers.

### Decision

The player × club × season grain will remain unchanged.

A separate unique-footballer recommendation view will be
introduced, selecting each candidate player's closest matching
historical spell.

The full spell-level ranking remains available for analysis.

This adjustment is a presentation and retrieval decision,
not a change to the underlying similarity metric.

Players with more eligible seasons have more opportunities to
produce a close match, which remains a limitation of this view.


## 3.3 Unique-Footballer Neighbour Rankings

The initial retrieval method ranks player-club-season spells
independently.

Consequently, several seasons of the same footballer can
appear among a target's closest neighbours.

For identifying alternative footballers, a complementary
unique-player ranking is constructed.

### Method

For a given target spell:

1. Calculate its distance to every eligible candidate spell.
2. Exclude all observations belonging to the target player.
3. For each remaining player_id, retain the spell with the
   smallest distance to the target.
4. Rank these candidate footballers by their minimum distance.

This approach preserves the historical spell-level representation
while preventing repeated seasons of one candidate from occupying
multiple recommendation positions.

It does not average player seasons or assume that a footballer's
style is constant throughout his career.

### Limitation

Players with more qualifying historical spells have more
opportunities to obtain a low minimum distance.

Thus, these rankings should be interpreted as each footballer's
closest observed historical style match, not necessarily his
current or typical playstyle.

In [ ]:
def get_spell_position(player, squad, season):

    mask = (
        (scaled_df["player"] == player)
        & (scaled_df["squad"] == squad)
        & (scaled_df["season"] == season)
    )

    positions = np.flatnonzero(mask.to_numpy())

    if len(positions) != 1:
        raise ValueError(
            f"Expected one spell for {player}, "
            f"{squad}, {season}. Found {len(positions)}."
        )

    return int(positions[0])


def find_similar_footballers(
    player,
    squad,
    season,
    k=10
):

    target_idx = get_spell_position(
        player, squad, season
    )

    # Candidate table follows distance-matrix row positions.
    candidates = (
        scaled_df[identity_cols]
        .reset_index(drop=True)
        .copy()
    )

    candidates["distance"] = distance_matrix[
        target_idx
    ]

    # Same-player observations are already masked.
    candidates = candidates[
        np.isfinite(candidates["distance"])
    ].copy()

    # Closest historical spell first
    candidates = candidates.sort_values(
        "distance",
        ascending=True,
        kind="mergesort"
    )

    # Retain one closest spell per candidate footballer
    candidates = candidates.drop_duplicates(
        subset="player_id",
        keep="first"
    ).head(k)

    assert candidates["player_id"].is_unique

    candidates.insert(
        0,
        "rank",
        np.arange(1, len(candidates) + 1)
    )

    return candidates[
        [
            "rank",
            "player",
            "squad",
            "season",
            "Minutes",
            "distance"
        ]
    ].reset_index(drop=True)


for player, squad, season in test_players:

    print("\n" + "=" * 65)
    print(f"{player} — {squad} ({season})")
    print("=" * 65)

    display(
        find_similar_footballers(
            player,
            squad,
            season,
            k=10
        ).round(3)
    )

In [ ]:
nearest_distance_population = np.min(
    distance_matrix,
    axis=1
)

target_diagnostics = []

for player, squad, season in test_players:

    idx = get_spell_position(
        player, squad, season
    )

    nearest_distance = nearest_distance_population[idx]

    percentile = (
        (nearest_distance_population <= nearest_distance)
        .mean() * 100
    )

    target_diagnostics.append({
        "player": player,
        "season": season,
        "nearest_distance": nearest_distance,
        "isolation_percentile": percentile
    })

target_diagnostics = pd.DataFrame(
    target_diagnostics
).sort_values(
    "isolation_percentile",
    ascending=False
)

display(target_diagnostics.round(2))


## 3.4 Findings — Unique-Footballer Similarity and Isolation

The unique-footballer retrieval method produces more diverse
candidate rankings without altering the underlying player-spell
representation.

### Jérémy Doku — Manchester City, 2023–24

Vinícius Júnior (2022–23) is the closest available footballer,
at a distance of 7.147.

Other relatively close candidates include Kaoru Mitoma,
Kylian Mbappé, Ousmane Dembélé and Franck Ribéry.

However, Doku has the highest nearest-neighbour distance
in the historical population, placing him at the 100th
isolation percentile.

His nearest available alternative should therefore not be
interpreted automatically as a close stylistic substitute.

### Mohamed Salah — Liverpool, 2017–18

Salah's closest candidates are Ansu Fati, Michail Antonio
and Serge Gnabry, with distances of 2.942, 3.024 and 3.041.

The narrow differences between these candidates suggest
that their precise ranking should not be overinterpreted.

The inclusion of Michail Antonio merits closer behavioural
inspection because the comparison is less immediately intuitive.

### Lionel Messi — Barcelona, 2018–19

Neymar (2019–20) is Messi's closest available historical
comparison, at a distance of 3.914.

Other candidates include Alexis Sánchez, Philippe Coutinho,
Ángel Di María, Cole Palmer and Rayan Cherki.

The broad profile of these footballers appears consistent
with Messi's progressive and creative attacking behaviour.

However, Messi's isolation percentile of 99.50 indicates
that his closest candidate is still relatively distant
compared with typical nearest-neighbour relationships.

### Overall Finding

All three test players are unusually isolated observations.

This reflects the deliberate selection of distinctive
attacking profiles and limits their representativeness.

A nearest-neighbour ranking identifies relative proximity
within the available historical population. It does not
guarantee that the closest candidate is sufficiently similar
for a particular recruitment or tactical application.

### Next Step

To investigate the football meaning of these comparisons,
we decompose Euclidean distance into individual feature
differences and examine shared behavioural characteristics.


# 4. Explaining Player-Style Similarity

## 4.1 Feature-Level Decomposition of Euclidean Distance

A single Euclidean distance summarizes differences across
all 20 behavioural features, but does not directly explain
which behaviours account for that distance.

Because Euclidean distance is based on squared standardized
differences, its squared value can be decomposed exactly:

$$
d(A,B)^2 = \sum_{j=1}^{20}(z_{Aj}-z_{Bj})^2
$$

For each feature, we calculate:

$$
C_j =
\frac{(z_{Aj}-z_{Bj})^2}{d(A,B)^2}\times100
$$

where \(C_j\) represents the feature's percentage contribution
to the squared distance between the two player spells.

The contributions sum to 100%.

### Interpretation

**Large contributions**

Features with large standardized differences account for
a substantial proportion of the pair's overall dissimilarity.

**Small contributions**

Features with small standardized differences contribute
relatively little to the pair's distance.

However, a small difference does not automatically imply
a meaningful shared playing tendency.

Two players may have similar values because both rarely
perform a particular action.

We therefore distinguish:

1. The largest behavioural differences.
2. Shared unusually high or low behavioural tendencies.

### Initial Comparisons

Three pairs are examined:

- Jérémy Doku (2023–24) vs Vinícius Júnior (2022–23).
- Mohamed Salah (2017–18) vs Michail Antonio (2019–20).
- Lionel Messi (2018–19) vs Neymar (2019–20).

These comparisons test both intuitive and less intuitive
nearest-neighbour relationships.

### Methodological Limitation

Feature contributions describe why two observations are
distant from one another.

They do not, by themselves, establish why one candidate
ranks ahead of every other candidate.

That requires a contrastive comparison against alternative
neighbours, which will be investigated separately.

In [ ]:
def explain_player_distance(target, candidate):

    target_idx = get_spell_position(*target)
    candidate_idx = get_spell_position(*candidate)

    # Standardized behavioural values
    z_target = X.iloc[target_idx].to_numpy(dtype=float)
    z_candidate = X.iloc[candidate_idx].to_numpy(dtype=float)

    # Original football-interpretable values
    raw_target = raw_df.iloc[target_idx][features].to_numpy(dtype=float)
    raw_candidate = raw_df.iloc[candidate_idx][features].to_numpy(dtype=float)

    # Exact squared Euclidean contributions
    z_gap = z_target - z_candidate
    squared_gap = z_gap ** 2

    squared_distance = squared_gap.sum()
    distance = np.sqrt(squared_distance)

    assert squared_distance > 0

    # Verify against existing distance matrix
    assert np.isclose(
        distance,
        distance_matrix[target_idx, candidate_idx]
    )

    family_map = feature_spec.set_index(
        "feature"
    )["macro_family"]

    explanation = pd.DataFrame({
        "feature": features,
        "macro_family": family_map.loc[features].to_numpy(),
        "raw_target": raw_target,
        "raw_candidate": raw_candidate,
        "z_target": z_target,
        "z_candidate": z_candidate,
        "absolute_z_gap": np.abs(z_gap),
        "squared_contribution": squared_gap,
        "contribution_pct": (
            squared_gap / squared_distance * 100
        )
    })

    # A shared extreme means both players are at least
    # one standard deviation from the historical mean
    # on the same side of the distribution.
    explanation["shared_extremity"] = np.where(
        z_target * z_candidate > 0,
        np.minimum(
            np.abs(z_target),
            np.abs(z_candidate)
        ),
        0
    )

    explanation = explanation.sort_values(
        "contribution_pct",
        ascending=False
    ).reset_index(drop=True)

    assert np.isclose(
        explanation["contribution_pct"].sum(),
        100
    )

    return explanation, distance

In [ ]:
comparison_cases = {
    "Doku vs Vinicius": (
        ("Jeremy Doku", "Manchester City", "2023-2024"),
        ("Vinicius Júnior", "Real Madrid", "2022-2023")
    ),

    "Salah vs Antonio": (
        ("Mohamed Salah", "Liverpool", "2017-2018"),
        ("Michail Antonio", "West Ham", "2019-2020")
    ),

    "Messi vs Neymar": (
        ("Lionel Messi", "Barcelona", "2018-2019"),
        ("Neymar", "Paris S-G", "2019-2020")
    )
}

comparison_explanations = {}

for name, (target, candidate) in comparison_cases.items():

    explanation, distance = explain_player_distance(
        target,
        candidate
    )

    comparison_explanations[name] = explanation

    print("\n" + "=" * 70)
    print(name)
    print(f"Euclidean distance: {distance:.3f}")
    print("=" * 70)

    print("\nLargest contributors to dissimilarity:")

    display(
        explanation[
            [
                "feature",
                "macro_family",
                "raw_target",
                "raw_candidate",
                "absolute_z_gap",
                "contribution_pct"
            ]
        ].head(8).round(3)
    )

    # Shared unusually high or low tendencies
    shared = (
        explanation[
            explanation["shared_extremity"] >= 1
        ]
        .sort_values(
            "shared_extremity",
            ascending=False
        )
    )

    print("\nShared extreme behavioural tendencies:")

    if shared.empty:
        print("No shared features exceed the diagnostic threshold.")
    else:
        display(
            shared[
                [
                    "feature",
                    "raw_target",
                    "raw_candidate",
                    "z_target",
                    "z_candidate"
                ]
            ].round(3)
        )

In [ ]:
explanation = comparison_explanations["Doku vs Vinicius"]

top_differences = (
    explanation.head(10)
    .sort_values("contribution_pct")
)

fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    top_differences["feature"],
    top_differences["contribution_pct"]
)

ax.set_xlabel("Contribution to Squared Distance (%)")
ax.set_title(
    "Doku vs Vinícius — Largest Behavioural Differences"
)

ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

print(
    "Top 10 features account for",
    f"{top_differences['contribution_pct'].sum():.1f}%",
    "of the squared distance."
)


## 4.2 Contrastive Explanations — Why One Player Ranks Ahead of Another

A pairwise distance decomposition identifies the features
responsible for differences between two player spells.

However, it does not explain why one candidate ranks ahead
of another.

For this, we compare the feature-level squared distances of
two candidates relative to the same target.

Let:

- T = target player spell
- A = higher-ranked candidate
- B = alternative candidate

For each feature:

$$
\Delta_j =
(z_{Tj}-z_{Bj})^2 -
(z_{Tj}-z_{Aj})^2
$$

The sum of these differences equals:

$$
\sum_j \Delta_j = d(T,B)^2 - d(T,A)^2
$$

### Interpretation

**Positive contribution**

Candidate A is closer to the target than candidate B on
that particular standardized feature.

**Negative contribution**

Candidate B is closer to the target than candidate A on
that feature.

The total explains the difference in squared Euclidean
distance between the two candidates.

This permits an exact, additive explanation of the ranking
under the selected 20-feature representation.

### Initial Cases

We investigate:

1. Why Vinícius Júnior ranks ahead of Kaoru Mitoma
   for Jérémy Doku.

2. Why Ansu Fati ranks ahead of Michail Antonio
   for Mohamed Salah.

These comparisons distinguish the features supporting the
higher-ranked candidate from those favouring the alternative.

Because correlated features may reinforce one another,
the resulting contributions remain conditional on the
chosen feature set and weighting scheme.

They explain the model's ranking, not causal football
mechanisms or recruitment suitability.

In [ ]:
def explain_candidate_ranking(target, candidate_a, candidate_b):

    t = get_spell_position(*target)
    a = get_spell_position(*candidate_a)
    b = get_spell_position(*candidate_b)

    z_t = X.iloc[t].to_numpy(dtype=float)
    z_a = X.iloc[a].to_numpy(dtype=float)
    z_b = X.iloc[b].to_numpy(dtype=float)

    sq_a = (z_t - z_a) ** 2
    sq_b = (z_t - z_b) ** 2

    # Positive = A is closer on this feature
    # Negative = B is closer on this feature
    advantage = sq_b - sq_a

    result = pd.DataFrame({
        "feature": features,
        "distance_sq_A": sq_a,
        "distance_sq_B": sq_b,
        "advantage_A": advantage
    })

    family_map = feature_spec.set_index(
        "feature"
    )["macro_family"]

    result["macro_family"] = (
        family_map.loc[features].to_numpy()
    )

    distance_a = np.sqrt(sq_a.sum())
    distance_b = np.sqrt(sq_b.sum())

    assert np.isclose(
        advantage.sum(),
        distance_b**2 - distance_a**2
    )

    assert distance_a < distance_b, (
        "Candidate A must be the closer candidate."
    )

    return result, distance_a, distance_b

In [ ]:
ranking_cases = {
    "Doku: Vinicius vs Mitoma": (
        ("Jeremy Doku", "Manchester City", "2023-2024"),
        ("Vinicius Júnior", "Real Madrid", "2022-2023"),
        ("Kaoru Mitoma", "Brighton", "2023-2024")
    ),

    "Salah: Fati vs Antonio": (
        ("Mohamed Salah", "Liverpool", "2017-2018"),
        ("Ansu Fati", "Barcelona", "2022-2023"),
        ("Michail Antonio", "West Ham", "2019-2020")
    )
}

ranking_explanations = {}

for name, (target, a, b) in ranking_cases.items():

    result, dist_a, dist_b = explain_candidate_ranking(
        target, a, b
    )

    ranking_explanations[name] = result

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    print(f"Candidate A distance: {dist_a:.3f}")
    print(f"Candidate B distance: {dist_b:.3f}")
    print(
        "Squared-distance advantage for A:",
        f"{dist_b**2 - dist_a**2:.3f}"
    )

    print("\nFeatures favouring Candidate A:")
    display(
        result.sort_values(
            "advantage_A", ascending=False
        ).head(7).round(3)
    )

    print("\nFeatures favouring Candidate B:")
    display(
        result.sort_values(
            "advantage_A", ascending=True
        ).head(7).round(3)
    )

In [ ]:
def plot_ranking_explanation(
    result,
    candidate_a_name,
    candidate_b_name,
    title,
    top_n=10
):

    top = (
        result.assign(
            magnitude=result["advantage_A"].abs()
        )
        .nlargest(top_n, "magnitude")
        .sort_values("advantage_A")
    )

    fig, ax = plt.subplots(figsize=(11, 7))

    colors = [
        "steelblue" if v > 0 else "darkorange"
        for v in top["advantage_A"]
    ]

    ax.barh(
        top["feature"],
        top["advantage_A"],
        color=colors
    )

    ax.axvline(0, color="black", linewidth=0.8)

    ax.set_xlabel(
        "Difference in Squared-Distance Contribution"
    )

    ax.set_title(title)

    ax.text(
        0.02, 1.02,
        f"← Favours {candidate_b_name}",
        transform=ax.transAxes,
        ha="left",
        fontsize=10
    )

    ax.text(
        0.98, 1.02,
        f"Favours {candidate_a_name} →",
        transform=ax.transAxes,
        ha="right",
        fontsize=10
    )

    ax.grid(axis="x", alpha=0.2)

    plt.tight_layout()
    plt.show()


plot_ranking_explanation(
    ranking_explanations["Doku: Vinicius vs Mitoma"],
    "Vinícius",
    "Mitoma",
    "Why Does Vinícius Rank Ahead of Mitoma for Doku?"
)

plot_ranking_explanation(
    ranking_explanations["Salah: Fati vs Antonio"],
    "Fati",
    "Antonio",
    "Why Does Fati Rank Ahead of Antonio for Salah?"
)


## 4.3 Findings — Contrastive Player Ranking Explanations

The contrastive analysis demonstrates that similarity rankings can
result from substantially different combinations of behavioural
advantages and disadvantages.

### Doku: Vinícius Júnior vs Kaoru Mitoma

Vinícius ranks ahead of Mitoma:

- Vinícius distance: 7.147
- Mitoma distance: 8.197
- Squared-distance advantage: 16.109

The largest advantages favouring Vinícius arise from carries into
the box and take-ons per 90.

Mitoma is closer to Doku on progressive-carry share, progressive
carry volume and passes into the box.

Thus, Vinícius is the better overall match under the selected
distance metric, although neither candidate closely reproduces
Doku's unusually extreme attacking profile.

### Salah: Ansu Fati vs Michail Antonio

The two candidates have nearly identical distances:

- Fati distance: 2.942
- Antonio distance: 3.024
- Squared-distance advantage: 0.486

Fati is closer on progressive receiving, shooting frequency and
several attacking progression measures.

Antonio is closer on crossing behaviour, territorial involvement
and long-pass share.

These opposing contributions largely offset one another.

The ranking is therefore not strongly separated under the
primary representation.

### General Interpretation

Contrastive explanations add information that a single similarity
score cannot provide.

They distinguish:

1. Candidates with a substantial overall distance advantage.
2. Candidates whose rankings depend on competing behavioural
   similarities.
3. The individual features supporting or opposing a ranking.

However, contributions are conditional on the feature set,
scaling and weighting scheme.

Correlated features can reinforce the same behavioural dimension.

The analysis establishes mathematical explanations of rankings,
not causal explanations of player performance or definitive
recruitment suitability.

### Next Step

We evaluate whether these rankings and their underlying
behavioural advantages remain consistent under family-balanced
feature weighting.


# 5. Similarity Robustness Analysis

## 5.1 Family-Weighting Sensitivity

The baseline similarity model uses Euclidean distance across
20 globally standardized behavioural features.

Under this representation, every feature contributes equally
before accounting for the observed differences between players.

However, behavioural families contain different numbers of
features, and correlated measurements can reinforce particular
attacking tendencies.

Notebook 04 constructed an alternative family-balanced
representation, weighting each feature by:

$$
w_j = \frac{1}{\sqrt{n_f}}
$$

where \(n_f\) is the number of retained features in its
behavioural family.

The corresponding squared distance is:

$$
d_w(A,B)^2 =
\sum_{j=1}^{20} w_j^2(z_{Aj}-z_{Bj})^2
$$

This balances the aggregate individual-feature variance
allocated to each family.

It does not eliminate within-family correlations.

### Research Questions

1. Does family balancing change the top-ten unique-footballer
   recommendations for our three test players?

2. Does Vinícius remain closer to Doku than Mitoma?

3. Does Fati remain closer to Salah than Antonio?

4. Are the previously observed differences robust, or do small
   ranking margins change under alternative weighting?

### Interpretation

A stable ranking provides evidence that the relationship does
not depend strongly on this weighting choice.

A ranking reversal indicates sensitivity to feature weighting,
not necessarily that either representation is incorrect.

Absolute distances from the two representations should not
be compared directly because the weighting changes the scale.

No representation will be declared superior solely because
its rankings appear more intuitively appealing.

In [ ]:
# Use the existing family-balanced representation from Notebook 04
X_balanced = balanced_df[features].copy()

weights = (
    feature_spec
    .set_index("feature")
    .loc[features, "family_weight"]
    .astype(float)
)

# Verify that family weights were applied correctly
expected_balanced = X.mul(weights, axis=1)

np.testing.assert_allclose(
    X_balanced.to_numpy(),
    expected_balanced.to_numpy(),
    rtol=1e-6,
    atol=1e-6
)

# Construct alternative distance matrix
balanced_distance_matrix = pairwise_distances(
    X_balanced.to_numpy(),
    metric="euclidean"
)

# Exclude all spells of the same footballer
player_ids = scaled_df["player_id"].to_numpy()

same_player = (
    player_ids[:, None] == player_ids[None, :]
)

balanced_distance_matrix[same_player] = np.inf

assert np.isfinite(
    balanced_distance_matrix[~same_player]
).all()

assert np.allclose(
    balanced_distance_matrix,
    balanced_distance_matrix.T
)

print("Family-balanced distance matrix validated.")
print("Shape:", balanced_distance_matrix.shape)

In [ ]:
def unique_player_ranking(target, distances):

    target_idx = get_spell_position(*target)

    candidates = (
        scaled_df[identity_cols]
        .reset_index(drop=True)
        .copy()
    )

    candidates["distance"] = distances[target_idx]

    candidates = candidates[
        np.isfinite(candidates["distance"])
    ].copy()

    candidates = (
        candidates
        .sort_values("distance", kind="mergesort")
        .drop_duplicates("player_id", keep="first")
        .reset_index(drop=True)
    )

    candidates.insert(
        0, "rank", np.arange(1, len(candidates) + 1)
    )

    return candidates


ranking_stability = []

for target in test_players:

    primary = unique_player_ranking(
        target, distance_matrix
    )

    balanced = unique_player_ranking(
        target, balanced_distance_matrix
    )

    # Compare top-ten unique footballers
    primary_ids = set(primary.head(10)["player_id"])
    balanced_ids = set(balanced.head(10)["player_id"])

    overlap = len(primary_ids & balanced_ids) / 10

    ranking_stability.append({
        "player": target[0],
        "season": target[2],
        "top10_overlap": overlap,
        "neighbours_changed": 10 - len(
            primary_ids & balanced_ids
        )
    })

    # Compare rankings of primary top-ten candidates
    comparison_table = primary.head(10)[
        [
            "player_id", "rank", "player",
            "squad", "season", "distance"
        ]
    ].merge(
        balanced[
            ["player_id", "rank", "season", "distance"]
        ],
        on="player_id",
        how="left",
        suffixes=("_primary", "_balanced")
    )

    print("\n" + "=" * 65)
    print(f"{target[0]} — {target[2]}")
    print(f"Top-10 unique-player overlap: {overlap:.0%}")
    print("=" * 65)

    display(
        comparison_table.drop(
            columns=["player_id"]
        ).round(3)
    )


ranking_stability_df = pd.DataFrame(
    ranking_stability
)

print("\nRanking stability summary:")
display(ranking_stability_df.round(3))

In [ ]:
head_to_head_results = []

for name, (target, a, b) in ranking_cases.items():

    t_idx = get_spell_position(*target)
    a_idx = get_spell_position(*a)
    b_idx = get_spell_position(*b)

    for representation, matrix in [
        ("Primary", distance_matrix),
        ("Family-balanced", balanced_distance_matrix)
    ]:

        dist_a = matrix[t_idx, a_idx]
        dist_b = matrix[t_idx, b_idx]

        advantage = dist_b**2 - dist_a**2

        head_to_head_results.append({
            "comparison": name,
            "representation": representation,
            "candidate_A_distance": dist_a,
            "candidate_B_distance": dist_b,
            "squared_advantage_A": advantage,
            "closer_candidate": (
                a[0] if advantage > 0
                else b[0] if advantage < 0
                else "Tie"
            )
        })

head_to_head_df = pd.DataFrame(
    head_to_head_results
)

display(head_to_head_df.round(3))


## 5.2 Initial Findings — Family-Weighting Sensitivity

The family-balanced representation retains 9 of the 10 closest
unique footballers for each of the three test players.

However, candidate-set agreement does not imply identical rankings.

### Jérémy Doku

The first three footballers remain Vinícius Júnior, Kaoru Mitoma
and Kylian Mbappé in both representations.

Vinícius also remains closer than Mitoma in the direct
head-to-head comparison.

This indicates that Doku's leading candidate relationships are
stable under the tested feature-family weighting.

However, his extreme isolation remains an important limitation.

### Mohamed Salah

The two representations identify the same five closest
footballers, but rank them differently.

Serge Gnabry moves from third to first, while Ansu Fati moves
from first to fifth.

The direct Fati-versus-Antonio comparison also reverses:
Fati is marginally closer under primary weighting, while
Antonio is marginally closer under family-balanced weighting.

These findings suggest that the candidate set is comparatively
stable, but precise rankings are sensitive to feature weighting.

### Lionel Messi

Neymar remains the highest-ranked footballer under both
representations.

However, the closest Neymar spell changes from 2019–20
to 2020–21.

This demonstrates that stability at the player-identity level
does not guarantee stability at the player-spell level.

### Preliminary Conclusion

The tested similarity relationships exhibit high candidate-set
stability under family balancing, with more substantial changes
in individual ranks and occasionally the selected historical spell.

The results support retaining both weighting schemes for
sensitivity analysis.

However, the three targets were deliberately selected from
unusually isolated observations. They cannot establish
population-wide stability.

The next analysis extends the robustness audit to the full
historical winger sample.


## 5.3 Population-Wide Neighbour Robustness

To assess whether the previous results generalize beyond three
selected footballers, the comparison is extended to all
1,396 player-club-season observations.

For each target spell, the ten closest unique footballers are
retrieved under both primary and family-balanced Euclidean
distance.

The analysis measures:

1. **Top-ten overlap:** The proportion of recommended footballers
   appearing in both representations.

2. **Top-one identity agreement:** Whether the same footballer
   is ranked first under both representations.

3. **Top-one spell agreement:** Whether the exact same historical
   player-club-season observation is ranked first.

4. **Ranking changes:** The average absolute rank movement among
   footballers shared by both top-ten lists.

The population-level distribution of these metrics is examined,
rather than relying only on their averages.

Because the dataset contains repeated observations of some
footballers across seasons, results are also summarized at
the unique target-player level.

This is an internal sensitivity analysis. Agreement between
representations does not establish external football validity.

In [ ]:
# Encode footballer identities for efficient comparisons
player_codes = pd.factorize(
    scaled_df["player_id"],
    sort=False
)[0]


def top_unique_indices(distances, codes, k=10):

    n = distances.shape[0]
    output = np.empty((n, k), dtype=int)

    for i in range(n):

        ordered = np.argsort(
            distances[i],
            kind="stable"
        )

        seen = {int(codes[i])}
        selected = []

        for j in ordered:

            if not np.isfinite(distances[i, j]):
                break

            candidate_id = int(codes[j])

            if candidate_id in seen:
                continue

            selected.append(j)
            seen.add(candidate_id)

            if len(selected) == k:
                break

        if len(selected) != k:
            raise ValueError(
                f"Insufficient candidates for row {i}"
            )

        output[i] = selected

    return output


primary_unique_idx = top_unique_indices(
    distance_matrix,
    player_codes,
    k=10
)

balanced_unique_idx = top_unique_indices(
    balanced_distance_matrix,
    player_codes,
    k=10
)

records = []

for i in range(len(scaled_df)):

    p_idx = primary_unique_idx[i]
    b_idx = balanced_unique_idx[i]

    p_ids = player_codes[p_idx]
    b_ids = player_codes[b_idx]

    p_ranks = {
        player_id: rank
        for rank, player_id in enumerate(p_ids, start=1)
    }

    b_ranks = {
        player_id: rank
        for rank, player_id in enumerate(b_ids, start=1)
    }

    shared = set(p_ranks) & set(b_ranks)

    mean_rank_change = np.mean([
        abs(p_ranks[pid] - b_ranks[pid])
        for pid in shared
    ]) if shared else np.nan

    records.append({
        "top10_overlap": len(shared) / 10,
        "same_top1_player": bool(
            p_ids[0] == b_ids[0]
        ),
        "same_top1_spell": bool(
            p_idx[0] == b_idx[0]
        ),
        "mean_rank_change_shared": mean_rank_change
    })


robustness_df = pd.concat(
    [
        scaled_df[
            ["player", "player_id", "squad", "season"]
        ].reset_index(drop=True),
        pd.DataFrame(records)
    ],
    axis=1
)

robustness_df["nearest_distance"] = (
    np.min(distance_matrix, axis=1)
)

summary = pd.DataFrame({
    "metric": [
        "Mean top-10 overlap",
        "Median top-10 overlap",
        "10th-percentile top-10 overlap",
        "Same top-ranked footballer (%)",
        "Same top-ranked spell (%)",
        "Mean shared-candidate rank change"
    ],
    "value": [
        robustness_df["top10_overlap"].mean(),
        robustness_df["top10_overlap"].median(),
        robustness_df["top10_overlap"].quantile(0.10),
        robustness_df["same_top1_player"].mean() * 100,
        robustness_df["same_top1_spell"].mean() * 100,
        robustness_df["mean_rank_change_shared"].mean()
    ]
})

print("Population-wide robustness:")
display(summary.round(3))

print("\nDistribution of top-10 overlap:")
display(
    robustness_df["top10_overlap"]
    .value_counts()
    .sort_index()
    .rename_axis("overlap")
    .reset_index(name="n_spells")
)

# Equal influence to each target footballer, rather than
# giving more influence to players with more eligible spells
player_level = robustness_df.groupby("player_id").agg(
    mean_overlap=("top10_overlap", "mean"),
    top1_agreement=("same_top1_player", "mean")
)

print("\nUnique-target-player-weighted summary:")
print(
    "Mean overlap:",
    round(player_level["mean_overlap"].mean(), 3)
)
print(
    "Top-1 agreement:",
    round(player_level["top1_agreement"].mean() * 100, 2),
    "%"
)

In [ ]:
# Exclude the three existing test footballers
tested_ids = {
    scaled_df.iloc[
        get_spell_position(*target)
    ]["player_id"]
    for target in test_players
}

eligible = robustness_df[
    ~robustness_df["player_id"].isin(tested_ids)
].copy()

selected_indices = []
selected_ids = set()

for percentile in [25, 50, 75]:

    target_distance = np.percentile(
        robustness_df["nearest_distance"],
        percentile
    )

    candidates = eligible.assign(
        deviation=lambda df: (
            df["nearest_distance"] - target_distance
        ).abs()
    ).sort_values("deviation")

    for idx, row in candidates.iterrows():

        if row["player_id"] not in selected_ids:
            selected_indices.append(idx)
            selected_ids.add(row["player_id"])
            break


representative_cases = robustness_df.loc[
    selected_indices
].copy()

display(
    representative_cases[
        [
            "player", "squad", "season",
            "nearest_distance", "top10_overlap",
            "same_top1_player", "mean_rank_change_shared"
        ]
    ].round(3)
)

for idx, row in representative_cases.iterrows():

    target = (
        row["player"],
        row["squad"],
        row["season"]
    )

    print("\n" + "=" * 60)
    print(f"{target[0]} — {target[1]} ({target[2]})")
    print("=" * 60)

    print("Primary top 5:")
    display(
        unique_player_ranking(
            target, distance_matrix
        )[
            ["rank", "player", "season", "distance"]
        ].head(5).round(3)
    )

    print("Family-balanced top 5:")
    display(
        unique_player_ranking(
            target, balanced_distance_matrix
        )[
            ["rank", "player", "season", "distance"]
        ].head(5).round(3)
    )


## 5.4 Findings — Population-Wide Family-Weighting Robustness

The population-wide analysis evaluates neighbour-ranking stability
across all 1,396 player-club-season observations.

### Candidate-Set Stability

Comparing the primary standardized and family-balanced
representations produces:

- Mean top-ten unique-footballer overlap: 86.2%
- Median top-ten overlap: 90%
- 10th-percentile overlap: 80%
- Same top-ranked footballer: 78.01%
- Same top-ranked historical spell: 77.22%
- Mean absolute rank change among shared candidates: 1.179 positions

Approximately 90.5% of target spells retain at least eight
of their ten recommended footballers.

The unique-target-player-weighted mean overlap (86.2%) is
nearly identical to the observation-weighted result.

This indicates substantial candidate-set stability under
the tested family-weighting alternative.

### Representative Player Validation

Three observations were selected near the 25th, 50th and
75th percentiles of nearest-neighbour distance.

**Gonçalo Guedes — Valencia, 2020–21**

The top two candidates, Daniel Podence and Levin Öztunalı,
remain unchanged across representations.

Top-ten overlap: 80%.

**Mathieu Dossevi — Toulouse, 2019–20**

David Ferreiro and Justin Kluivert remain the two closest
candidates. The top five footballers are unchanged as a set.

Top-ten overlap: 80%.

**Ismaïla Sarr — Rennes, 2018–19**

Samuel Chukwueze remains the closest footballer, although
his best-matching historical season changes.

Top-ten overlap: 60%.

Sarr demonstrates that broader neighbour rankings may change
even when the highest-ranked footballer remains stable.

### Interpretation

The similarity model demonstrates substantial internal
robustness to family weighting across the historical population.

However, candidate identity, candidate ranking and selected
historical spell exhibit different degrees of stability.

Robustness is not uniform across player profiles.

Agreement between representations establishes consistency
under the tested assumptions, not independent football validity.

### Methodological Decision

The complete 20-feature standardized representation remains
the primary similarity baseline.

The family-balanced representation will be retained as a
sensitivity alternative.

Before finalizing the distance metric, we investigate whether
replacing squared-difference-based Euclidean distance with
absolute-difference-based Manhattan distance materially
changes the recommendations.


## 5.5 Distance Metric Sensitivity — Euclidean vs Manhattan

The previous robustness analysis examined alternative
feature-family weighting while retaining Euclidean distance.

However, similarity rankings also depend on how feature
differences are aggregated.

### Euclidean Distance (L2)

$$
d_{L2}(A,B)=\sqrt{\sum_j(z_{Aj}-z_{Bj})^2}
$$

Euclidean distance gives relatively greater influence to
large individual standardized differences.

### Manhattan Distance (L1)

$$
d_{L1}(A,B)=\sum_j|z_{Aj}-z_{Bj}|
$$

Manhattan distance aggregates absolute standardized
differences without squaring them.

It therefore responds differently to profiles containing
a small number of extreme feature differences.

### Motivation

Earlier pairwise analysis showed that unusually large
carrying differences dominated Doku's Euclidean distance
from his closest candidates.

Manhattan distance provides a reasonable alternative for
investigating sensitivity to this behaviour.

### Analytical Questions

1. How much do unique-footballer recommendations change
   when Euclidean distance is replaced by Manhattan distance?

2. Does the identity of the top-ranked candidate remain
   stable across the population?

3. Are highly isolated players particularly sensitive
   to the alternative distance definition?

4. Do representative player profiles maintain broadly
   consistent recommendations?

### Method

Both metrics are calculated on the same globally standardized
20-feature representation.

No feature engineering, rescaling or dimensionality reduction
is performed.

Same-player observations are excluded, and candidate
footballers are deduplicated using their closest matching spell.

Absolute distances are not directly compared between metrics,
as L1 and L2 have different numerical scales.

This analysis is a sensitivity check, not a search for a
distance metric that produces more famous or familiar
footballer comparisons.

In [ ]:

# Manhattan distance on the SAME standardized feature matrix
manhattan_distance_matrix = pairwise_distances(
    X.to_numpy(),
    metric="manhattan"
)

# Preserve the same-player exclusion
manhattan_distance_matrix[same_player] = np.inf

assert manhattan_distance_matrix.shape == distance_matrix.shape

assert np.isfinite(
    manhattan_distance_matrix[~same_player]
).all()

assert np.allclose(
    manhattan_distance_matrix,
    manhattan_distance_matrix.T
)

# Retrieve top-ten distinct footballers under Manhattan distance
manhattan_unique_idx = top_unique_indices(
    manhattan_distance_matrix,
    player_codes,
    k=10
)


def compare_neighbour_rankings(reference_idx, alternative_idx):

    results = []

    for i in range(len(reference_idx)):

        ref_spells = reference_idx[i]
        alt_spells = alternative_idx[i]

        ref_ids = player_codes[ref_spells]
        alt_ids = player_codes[alt_spells]

        ref_ranks = {
            pid: rank
            for rank, pid in enumerate(ref_ids, 1)
        }

        alt_ranks = {
            pid: rank
            for rank, pid in enumerate(alt_ids, 1)
        }

        shared = set(ref_ranks) & set(alt_ranks)

        mean_rank_change = (
            np.mean([
                abs(ref_ranks[pid] - alt_ranks[pid])
                for pid in shared
            ])
            if shared else np.nan
        )

        results.append({
            "top10_overlap": len(shared) / 10,
            "same_top1_player": ref_ids[0] == alt_ids[0],
            "same_top1_spell": ref_spells[0] == alt_spells[0],
            "mean_shared_rank_change": mean_rank_change
        })

    return pd.DataFrame(results)


# Compare both alternative methods to primary Euclidean
balanced_agreement = compare_neighbour_rankings(
    primary_unique_idx,
    balanced_unique_idx
)

manhattan_agreement = compare_neighbour_rankings(
    primary_unique_idx,
    manhattan_unique_idx
)


def summarize_agreement(label, df):

    return {
        "representation": label,
        "mean_top10_overlap": df["top10_overlap"].mean(),
        "median_top10_overlap": df["top10_overlap"].median(),
        "p10_top10_overlap": df["top10_overlap"].quantile(0.1),
        "same_top1_player_pct": (
            df["same_top1_player"].mean() * 100
        ),
        "same_top1_spell_pct": (
            df["same_top1_spell"].mean() * 100
        ),
        "mean_shared_rank_change": (
            df["mean_shared_rank_change"].mean()
        )
    }


metric_comparison = pd.DataFrame([
    summarize_agreement(
        "Family-balanced Euclidean",
        balanced_agreement
    ),
    summarize_agreement(
        "Standardized Manhattan",
        manhattan_agreement
    )
])

display(metric_comparison.round(3))

print("\nManhattan top-10 overlap distribution:")

display(
    manhattan_agreement["top10_overlap"]
    .value_counts()
    .sort_index()
    .rename_axis("overlap")
    .reset_index(name="n_spells")
)

In [ ]:

inspection_targets = test_players + [
    (
        row["player"],
        row["squad"],
        row["season"]
    )
    for _, row in representative_cases.iterrows()
]

individual_metric_results = []

for target in inspection_targets:

    primary = unique_player_ranking(
        target,
        distance_matrix
    )

    manhattan = unique_player_ranking(
        target,
        manhattan_distance_matrix
    )

    primary_top5 = set(
        primary.head(5)["player_id"]
    )

    manhattan_top5 = set(
        manhattan.head(5)["player_id"]
    )

    primary_winner_id = primary.iloc[0]["player_id"]

    # Locate the primary winner under Manhattan ranking
    new_rank = manhattan.loc[
        manhattan["player_id"] == primary_winner_id,
        "rank"
    ].iloc[0]

    individual_metric_results.append({
        "target": target[0],
        "season": target[2],
        "euclidean_top1": primary.iloc[0]["player"],
        "manhattan_top1": manhattan.iloc[0]["player"],
        "primary_winner_L1_rank": new_rank,
        "top5_overlap": (
            len(primary_top5 & manhattan_top5) / 5
        )
    })


individual_metric_df = pd.DataFrame(
    individual_metric_results
)

display(individual_metric_df.round(3))


## 5.6 Findings — Distance Metric Sensitivity

The Manhattan distance sensitivity analysis indicates substantial
agreement with the primary Euclidean representation, alongside
meaningful changes in individual rankings.

### Population-Wide Results

Compared with standardized Euclidean distance, Manhattan produces:

- Mean top-ten unique-footballer overlap: 79.0%
- Median top-ten overlap: 80%
- 10th-percentile overlap: 70%
- Same top-ranked footballer: 69.34%
- Same top-ranked historical spell: 68.41%
- Mean absolute rank change among shared candidates: 1.586 positions

Approximately 91.4% of target spells retain at least seven of
their ten recommended footballers.

However, the highest-ranked player changes for approximately
30.7% of observations.

This indicates that the overall candidate population is fairly
consistent, but precise rankings are sensitive to the choice
of distance metric.

### Individual Player Examples

Vinícius Júnior remains the closest candidate for Jérémy Doku,
while Neymar remains closest to Lionel Messi.

Daniel Podence, David Ferreiro and Samuel Chukwueze also retain
their respective first-place positions for Gonçalo Guedes,
Mathieu Dossevi and Ismaïla Sarr.

For Mohamed Salah, the highest-ranked candidate changes from
Ansu Fati under Euclidean distance to Serge Gnabry under
Manhattan distance.

Across the six inspected targets, top-five agreement ranges
from 60% to 80%.

### Interpretation

Euclidean distance places relatively greater emphasis on large
individual standardized differences.

Manhattan distance instead aggregates absolute feature gaps,
making it less dominated by individual extreme mismatches.

Neither metric is inherently superior for identifying
football playstyles.

Their agreement provides evidence of consistency across two
reasonable distance definitions.

Their disagreement identifies player comparisons whose rankings
depend on how behavioural differences are aggregated.

### Provisional Methodological Decision

Standardized Euclidean distance remains the primary baseline
because it is transparent, geometrically interpretable and
supports exact feature-level squared-distance explanations.

Manhattan distance is retained as a robustness alternative.

This choice does not establish that Euclidean distance is
objectively more accurate or better at identifying tactical
replacements.

The remaining targeted diagnostic investigates the Salah
ranking reversal before finalizing the similarity methodology.


## 5.7 Case Study — Why Distance Metrics Disagree for Salah

Mohamed Salah's 2017–18 Liverpool profile provides an example
where the highest-ranked candidate changes with the distance
definition.

Under standardized Euclidean distance:

- Ansu Fati (2022–23) ranks first.
- Serge Gnabry (2019–20) ranks third.

Under standardized Manhattan distance:

- Serge Gnabry ranks first.
- Ansu Fati ranks second.

To explain this reversal, we compare the same historical
candidate spells against Salah using both metrics.

For each feature, the contrastive advantage favouring Fati is:

**Euclidean squared-distance advantage**

$$
\Delta_j^{L2} =
(z_{Sj}-z_{Gj})^2 -
(z_{Sj}-z_{Fj})^2
$$

**Manhattan distance advantage**

$$
\Delta_j^{L1} =
|z_{Sj}-z_{Gj}| -
|z_{Sj}-z_{Fj}|
$$

Here, S represents Salah, F represents Fati and G
represents Gnabry.

Positive values favour Fati; negative values favour Gnabry.

Summing the feature contributions produces the exact
corresponding distance advantage.

The objective is to identify which behavioural differences
explain the change in ranking, without assuming either
metric is objectively preferable.

In [ ]:
# Exact historical spells
target = (
    "Mohamed Salah",
    "Liverpool",
    "2017-2018"
)

candidate_fati = (
    "Ansu Fati",
    "Barcelona",
    "2022-2023"
)

candidate_gnabry = (
    "Serge Gnabry",
    "Bayern Munich",
    "2019-2020"
)

t = get_spell_position(*target)
f = get_spell_position(*candidate_fati)
g = get_spell_position(*candidate_gnabry)

# Standardized feature vectors
z_t = X.iloc[t].to_numpy(dtype=float)
z_f = X.iloc[f].to_numpy(dtype=float)
z_g = X.iloc[g].to_numpy(dtype=float)

gap_f = np.abs(z_t - z_f)
gap_g = np.abs(z_t - z_g)

# Positive advantage favours Fati
advantage_l2 = gap_g**2 - gap_f**2
advantage_l1 = gap_g - gap_f

comparison_df = pd.DataFrame({
    "feature": features,
    "Salah_Fati_abs_gap": gap_f,
    "Salah_Gnabry_abs_gap": gap_g,
    "advantage_Fati_L2_squared": advantage_l2,
    "advantage_Fati_L1": advantage_l1
})

# Exact distance validations
fati_l2 = np.linalg.norm(z_t - z_f)
gnabry_l2 = np.linalg.norm(z_t - z_g)

fati_l1 = gap_f.sum()
gnabry_l1 = gap_g.sum()

assert np.isclose(
    advantage_l2.sum(),
    gnabry_l2**2 - fati_l2**2
)

assert np.isclose(
    advantage_l1.sum(),
    gnabry_l1 - fati_l1
)

assert np.isclose(
    fati_l2, distance_matrix[t, f]
)

assert np.isclose(
    gnabry_l1, manhattan_distance_matrix[t, g]
)

print("EUCLIDEAN DISTANCES")
print(f"Fati:   {fati_l2:.3f}")
print(f"Gnabry: {gnabry_l2:.3f}")

print("\nMANHATTAN DISTANCES")
print(f"Fati:   {fati_l1:.3f}")
print(f"Gnabry: {gnabry_l1:.3f}")

print("\nTOTAL CONTRASTIVE ADVANTAGES")
print(
    "Euclidean squared advantage for Fati:",
    round(advantage_l2.sum(), 3)
)
print(
    "Manhattan advantage for Fati:",
    round(advantage_l1.sum(), 3)
)

print("\nFeatures most favouring Fati under L2:")
display(
    comparison_df.sort_values(
        "advantage_Fati_L2_squared",
        ascending=False
    ).head(6).round(3)
)

print("\nFeatures most favouring Gnabry under L1:")
display(
    comparison_df.sort_values(
        "advantage_Fati_L1",
        ascending=True
    ).head(6).round(3)
)

print("\nLargest changes in relative feature influence:")
display(
    comparison_df.assign(
        abs_l2_advantage=lambda d:
            d["advantage_Fati_L2_squared"].abs(),
        abs_l1_advantage=lambda d:
            d["advantage_Fati_L1"].abs()
    ).sort_values(
        "abs_l2_advantage",
        ascending=False
    ).head(10).round(3)
)

In [ ]:
target = ("Mohamed Salah", "Liverpool", "2017-2018")

display(
    unique_player_ranking(
        target,
        manhattan_distance_matrix
    )[
        ["rank", "player", "squad", "season", "distance"]
    ].head(5)
)


## 5.8 Reconciling Historical Spell Selection

The initial Salah–Fati–Gnabry metric comparison identified an
important distinction between fixed-spell comparisons and
unique-footballer recommendations.

A footballer may be represented by several qualifying seasons.

The unique-footballer retrieval procedure independently selects
the closest historical spell for each candidate under each
distance metric.

Consequently, the closest season of a footballer can change
when the metric changes.

The initial fixed-spell Manhattan comparison showed that
Ansu Fati (2022–23) was closer to Salah than Serge Gnabry
(2019–20).

However, the unique-footballer Manhattan ranking placed
Gnabry ahead of Fati.

To reconcile these results, the analysis retrieves the exact
historical spells selected under each metric.

It then compares feature-level distances using the spells
actually chosen by the Manhattan ranking.

This distinction is essential for interpreting player-level
recommendations without conflating different historical
observations of the same footballer.

In [ ]:
target = (
    "Mohamed Salah",
    "Liverpool",
    "2017-2018"
)

fati_id = scaled_df.iloc[
    get_spell_position(
        "Ansu Fati", "Barcelona", "2022-2023"
    )
]["player_id"]

gnabry_id = scaled_df.iloc[
    get_spell_position(
        "Serge Gnabry", "Bayern Munich", "2019-2020"
    )
]["player_id"]

selection_rows = []

for metric_name, matrix in [
    ("Euclidean", distance_matrix),
    ("Manhattan", manhattan_distance_matrix)
]:

    ranking = unique_player_ranking(target, matrix)

    for name, pid in [
        ("Ansu Fati", fati_id),
        ("Serge Gnabry", gnabry_id)
    ]:

        matched = ranking.loc[
            ranking["player_id"] == pid
        ].iloc[0]

        selection_rows.append({
            "metric": metric_name,
            "candidate": name,
            "rank": int(matched["rank"]),
            "matched_squad": matched["squad"],
            "matched_season": matched["season"],
            "distance": matched["distance"]
        })

spell_selection_audit = pd.DataFrame(
    selection_rows
)

display(spell_selection_audit.round(3))

In [ ]:
manhattan_ranking = unique_player_ranking(
    target,
    manhattan_distance_matrix
)

fati_match = manhattan_ranking.loc[
    manhattan_ranking["player_id"] == fati_id
].iloc[0]

gnabry_match = manhattan_ranking.loc[
    manhattan_ranking["player_id"] == gnabry_id
].iloc[0]


def position_from_match(row):
    return get_spell_position(
        row["player"],
        row["squad"],
        row["season"]
    )


t = get_spell_position(*target)
f = position_from_match(fati_match)
g = position_from_match(gnabry_match)

z_t = X.iloc[t].to_numpy(dtype=float)
z_f = X.iloc[f].to_numpy(dtype=float)
z_g = X.iloc[g].to_numpy(dtype=float)

gap_f = np.abs(z_t - z_f)
gap_g = np.abs(z_t - z_g)

corrected_metric_explanation = pd.DataFrame({
    "feature": features,
    "Fati_abs_gap": gap_f,
    "Gnabry_abs_gap": gap_g,

    # Positive values favour Fati
    "Fati_advantage_L2_squared": gap_g**2 - gap_f**2,
    "Fati_advantage_L1": gap_g - gap_f
})

# Validation against actual Manhattan rankings
assert np.isclose(
    gap_f.sum(),
    manhattan_distance_matrix[t, f]
)

assert np.isclose(
    gap_g.sum(),
    manhattan_distance_matrix[t, g]
)

assert np.isclose(
    corrected_metric_explanation["Fati_advantage_L1"].sum(),
    gap_g.sum() - gap_f.sum()
)

assert np.isclose(
    corrected_metric_explanation[
        "Fati_advantage_L2_squared"
    ].sum(),
    np.sum(gap_g**2) - np.sum(gap_f**2)
)

print(
    "Fati matched season:",
    fati_match["season"]
)
print(
    "Gnabry matched season:",
    gnabry_match["season"]
)

print("\nManhattan:")
print("Fati:", round(gap_f.sum(), 3))
print("Gnabry:", round(gap_g.sum(), 3))

print("\nSame spells evaluated with Euclidean:")
print("Fati:", round(np.linalg.norm(gap_f), 3))
print("Gnabry:", round(np.linalg.norm(gap_g), 3))

print("\nFeatures favouring Gnabry under Manhattan:")
display(
    corrected_metric_explanation
    .sort_values("Fati_advantage_L1")
    .head(7)
    .round(3)
)

print("\nFeatures favouring Fati under Manhattan:")
display(
    corrected_metric_explanation
    .sort_values(
        "Fati_advantage_L1",
        ascending=False
    )
    .head(7)
    .round(3)
)


## 5.9 Final Diagnostic — Metric-Dependent Ranking Reversal

The Salah–Fati–Gnabry comparison resolves the discrepancy
between the initial fixed-spell diagnostic and the
unique-footballer Manhattan rankings.

### Historical Spell Selection

The closest Gnabry observation changes between metrics:

- Euclidean: Bayern Munich, 2019–20.
- Manhattan: Bayern Munich, 2022–23.

Fati's matched observation remains Barcelona, 2022–23.

This demonstrates that unique-footballer rankings must preserve
the specific historical spell selected under each metric.

### Fixed-Spell Comparison

Comparing Fati 2022–23 and Gnabry 2022–23 against
Salah 2017–18:

| Candidate | Euclidean | Manhattan |
|---|---:|---:|
| Ansu Fati | 2.942 | 10.706 |
| Serge Gnabry | 3.278 | 10.174 |

Fati is closer under Euclidean distance, while Gnabry
is closer under Manhattan distance.

The reversal therefore persists even when comparing
the same historical observations.

### Feature-Level Explanation

Fati benefits particularly from a smaller mismatch in
attacking-third touch share and progressive receiving.

Gnabry benefits from smaller differences across several
other behaviours, including:

- Long-pass share.
- Crossing volume and propensity.
- Progressive passing share.
- Shooting propensity.
- Progressive carrying share.

Euclidean distance places greater influence on the large
territorial mismatch affecting Gnabry.

Manhattan distance aggregates the smaller absolute
differences across multiple features, producing a
different overall ranking.

### Methodological Conclusion

The observed reversal results from the interaction of:

1. Distance metric choice.
2. Feature-level differences.
3. Selection of the closest historical candidate spell.

The similarity framework is therefore sensitive to
methodological assumptions, particularly for candidates
with competing behavioural strengths.

Standardized Euclidean distance is retained as the
primary baseline, with Manhattan and family-balanced
Euclidean distance retained for robustness analysis.

No further distance-metric experimentation is required
at this stage.


# 6. Conclusions and Downstream Implications

## 6.1 Summary of Findings

This notebook developed an explainable player-style similarity
framework using 1,396 historical winger player-club-season
observations and 20 standardized behavioural features.

### 1. Style Similarity Can Be Measured Without Player Quality

Euclidean distance provides a transparent baseline for
comparing attacking behaviours independently of goals,
assists, finishing efficiency and other performance outcomes.

The analysis measures similarity of observed attacking
activity, not overall footballing ability or tactical
interchangeability.

### 2. Nearest Neighbours Are Not Necessarily Close Matches

The population median nearest-neighbour distance is 1.982.

However, some exceptionally distinctive player spells have
considerably larger nearest-neighbour distances.

Jérémy Doku's 2023–24 Manchester City spell has the highest
nearest-neighbour distance in the dataset (7.147).

This demonstrates why rankings must be interpreted alongside
absolute distance and population-relative isolation.

### 3. Unique-Footballer Recommendations Improve Interpretability

For player-comparison applications, each candidate footballer
is represented by the closest matching historical spell.

This avoids repeatedly recommending different seasons of the
same player while preserving the original observational grain.

However, candidates with more qualifying seasons have more
opportunities to produce a close historical match.

### 4. Similarity Comparisons Are Explainable

The model supports two complementary explanations:

**Pairwise decomposition**

Identifies the features contributing most strongly to the
squared Euclidean distance between two player spells.

**Contrastive decomposition**

Identifies the features explaining why one candidate ranks
ahead of another under the chosen metric.

These explanations are exact mathematical decompositions
under the specified feature representation.

They should not be interpreted as causal explanations of
football performance.

### 5. Family-Weighting Robustness Is Substantial

Comparing primary and family-balanced Euclidean distances:

- Mean top-ten unique-footballer overlap: 86.2%.
- Same top-ranked footballer: 78.01%.
- Same top-ranked historical spell: 77.22%.

This indicates substantial candidate-set stability under the
tested alternative weighting, although individual ranks
can change.

### 6. Distance Metric Choice Remains Relevant

Comparing standardized Euclidean and Manhattan distances:

- Mean top-ten unique-footballer overlap: 79.0%.
- Same top-ranked footballer: 69.34%.
- Same top-ranked historical spell: 68.41%.

The metric changes the emphasis placed on concentrated
extreme mismatches versus differences distributed across
multiple features.

Candidate rankings must therefore be interpreted as
conditional on the selected distance definition.

## 6.2 Final Methodological Decisions

**Primary representation**

20 globally standardized attacking-behaviour features.

**Primary distance metric**

Euclidean distance.

**Sensitivity alternatives**

- Family-balanced Euclidean distance.
- Standardized Manhattan distance.

**Recommendation format**

Unique footballers, represented by their closest matching
historical player-club-season spell.

**Explanation methodology**

- Feature-level squared-distance contributions.
- Shared unusually high/low behavioural tendencies.
- Contrastive feature contributions to candidate rankings.

These choices provide a transparent and reproducible
baseline, rather than a proven optimal measure of football
similarity.

## 6.3 Limitations

1. Aggregate season-spell statistics are influenced by team
   tactics, possession, role and competitive context.

2. Correlated retained features can reinforce particular
   attacking behaviours.

3. The representation covers selected attacking tendencies,
   not defensive contributions, tactical intelligence,
   off-ball movement in full detail or player quality.

4. Closest historical matches are not necessarily current
   or realistic recruitment alternatives.

5. Multiple qualifying seasons give some candidate players
   more opportunities to achieve a small minimum distance.

6. Some historical feature definitions and provider changes
   may affect cross-season comparability.

7. Internal robustness does not establish independent
   tactical validity.

## 6.4 Next Research Stage

The next stage investigates whether the continuous
20-dimensional attacking-style space contains meaningful
player groupings.

Clustering will be treated as an empirical question,
not an assumption.

Any identified archetypes must be interpreted using
behavioural features, representative players, within-group
variation and stability diagnostics.

If the population does not support defensible discrete
groups, continuous playstyle profiles and nearest-neighbour
similarity will remain valid principal project outputs.

Quality/output measures will remain outside the initial
archetype-discovery process.

In [ ]:
import json

OUTPUT_06 = OUTPUT_DIR / "notebook_06_outputs"
OUTPUT_06.mkdir(parents=True, exist_ok=True)

n = len(scaled_df)
k = 10

# Create reusable, fully identified neighbour tables
def export_neighbours(indices, distances):

    target_rows = np.repeat(np.arange(n), k)
    candidate_rows = indices.reshape(-1)

    result = pd.DataFrame({
        "target_row_idx": target_rows,
        "candidate_row_idx": candidate_rows,
        "rank": np.tile(np.arange(1, k + 1), n),
        "distance": distances[
            target_rows, candidate_rows
        ]
    })

    for col in [
        "player", "player_id", "squad",
        "season", "Minutes"
    ]:

        values = scaled_df[col].to_numpy()

        result[f"target_{col}"] = values[target_rows]
        result[f"candidate_{col}"] = values[candidate_rows]

    assert len(result) == n * k

    assert (
        result["target_player_id"]
        != result["candidate_player_id"]
    ).all()

    return result


representations = {
    "primary_euclidean": (
        primary_unique_idx, distance_matrix
    ),
    "family_balanced_euclidean": (
        balanced_unique_idx, balanced_distance_matrix
    ),
    "standardized_manhattan": (
        manhattan_unique_idx, manhattan_distance_matrix
    )
}

for name, (indices, distances) in representations.items():

    table = export_neighbours(indices, distances)

    table.to_parquet(
        OUTPUT_06 / f"top10_{name}.parquet",
        index=False
    )

# Robustness and diagnostic outputs
robustness_df.to_parquet(
    OUTPUT_06 / "family_weighting_robustness.parquet",
    index=False
)

metric_comparison.to_csv(
    OUTPUT_06 / "distance_metric_comparison.csv",
    index=False
)

target_diagnostics.to_csv(
    OUTPUT_06 / "initial_target_isolation.csv",
    index=False
)

spell_selection_audit.to_csv(
    OUTPUT_06 / "salah_spell_selection_audit.csv",
    index=False
)

corrected_metric_explanation.to_csv(
    OUTPUT_06 / "salah_metric_explanation.csv",
    index=False
)

# Existing pairwise and contrastive explanations
pairwise_explanations_export = pd.concat(
    comparison_explanations,
    names=["comparison", "row"]
).reset_index(level=0).reset_index(drop=True)

pairwise_explanations_export.to_csv(
    OUTPUT_06 / "pairwise_feature_explanations.csv",
    index=False
)

contrastive_explanations_export = pd.concat(
    ranking_explanations,
    names=["comparison", "row"]
).reset_index(level=0).reset_index(drop=True)

contrastive_explanations_export.to_csv(
    OUTPUT_06 / "contrastive_ranking_explanations.csv",
    index=False
)

# Reproducibility metadata
manifest = {
    "notebook": "06_player_style_similarity",
    "source_notebook": "04_playstyle_feature_engineering",
    "n_observations": n,
    "n_features": len(features),
    "features": features,
    "observational_grain": "player x club x season",
    "primary_representation": "globally standardized",
    "primary_metric": "euclidean",
    "sensitivity_representations": [
        "family_balanced_euclidean",
        "standardized_manhattan"
    ],
    "same_player_excluded": True,
    "recommendation_unit": "unique player_id",
    "candidate_selection": "closest eligible historical spell",
    "top_k": k,
    "quality_outputs_excluded": True,
    "note": (
        "Distances are representation-specific. "
        "Rankings describe attacking behaviour, not player quality."
    )
}

with open(
    OUTPUT_06 / "similarity_manifest.json",
    "w"
) as f:
    json.dump(manifest, f, indent=2)

print("Notebook 06 exports completed:\n")

for path in sorted(OUTPUT_06.iterdir()):
    print(path.name)

In [ ]:
required_exports = [
    "top10_primary_euclidean.parquet",
    "top10_family_balanced_euclidean.parquet",
    "top10_standardized_manhattan.parquet",
    "family_weighting_robustness.parquet",
    "distance_metric_comparison.csv",
    "initial_target_isolation.csv",
    "salah_spell_selection_audit.csv",
    "salah_metric_explanation.csv",
    "pairwise_feature_explanations.csv",
    "contrastive_ranking_explanations.csv",
    "similarity_manifest.json"
]

missing = [
    file for file in required_exports
    if not (OUTPUT_06 / file).exists()
]

assert not missing, f"Missing exports: {missing}"

for name in [
    "primary_euclidean",
    "family_balanced_euclidean",
    "standardized_manhattan"
]:
    df = pd.read_parquet(
        OUTPUT_06 / f"top10_{name}.parquet"
    )

    assert len(df) == 1396 * 10

    assert (
        df["target_player_id"]
        != df["candidate_player_id"]
    ).all()

    assert (
        df.groupby("target_row_idx")["candidate_player_id"]
        .nunique()
        == 10
    ).all()

print("All Notebook 06 exports validated.")
print(f"Output directory: {OUTPUT_06}")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('notebook_06_outputs', 'zip', OUTPUT_DIR/'notebook_06_outputs')

files.download('notebook_06_outputs.zip')